In [76]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from tqdm import tqdm

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
}

In [5]:
def get_html(url):
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    return resp.text

In [62]:
def scrape_player_bio(player_id):
    # name, height_cm, weight_kg, position, shoots, college, draft_year
    # birth_date, birth_place, ws, per
 
    first_letter = player_id[0]
    url = "https://www.basketball-reference.com/players/" + first_letter + "/" + player_id + ".html"
 
    html = get_html(url)
    soup = BeautifulSoup(html, "html.parser")
 
    meta_div = soup.find("div", {"id": "meta"})
 
    result = {"player_id": player_id}

    name_tag = meta_div.find("h1")
    if name_tag is not None:
        result["name"] = name_tag.get_text(strip=True)
 
    paragraphs = meta_div.find_all("p")
 
    for p in paragraphs:
        text = p.get_text(" ", strip=True)
 
        # 6-8, 237lb (203cm, 107kg)
        if "cm" in text and "kg" in text:
            match = re.search(r"\((\d+)cm,\s*(\d+)kg\)", text)
            if match:
                result["height_cm"] = int(match.group(1))
                result["weight_kg"] = int(match.group(2))
 
        # Position and shoot  
        if "Position:" in text:
            after_position = text.split("Position:")[1]
            result["position"] = after_position.split("▪")[0].strip()
 
            if "Shoots:" in text:
                after_shoots = text.split("Shoots:")[1].strip()
                result["shoots"] = after_shoots.split()[0]
 
        # College
        if "College:" in text:
            college_link = p.find("a")
            if college_link is not None:
                result["college"] = college_link.get_text(strip=True)
 
        # Draft
        if "Draft:" in text:
            match = re.search(r"(\d{4}) NBA Draft", text)
            if match:
                result["draft_year"] = int(match.group(1))

        # NBA Debut
        if "NBA Debut:" in text:
            debut_link = p.find("a")
            if debut_link is not None:
                result["nba_debut"] = debut_link.get_text(strip=True)
 
    # birth date and birth place same span
    birth_span = meta_div.find("span", {"id": "necro-birth"})
    if birth_span is not None:
        result["birth_date"] = birth_span.get_text(strip=True)
 
        birth_p = birth_span.find_parent("p")
        links_in_p = birth_p.find_all("a")
        if links_in_p:
            result["birth_place"] = links_in_p[-1].get_text(strip=True)

    # WS and PER
    summary_div = soup.find("div", {"class": "stats_pullout"})
    if summary_div is not None:
        stat_spans = summary_div.find_all("span", {"class": "poptip"})

        for span in stat_spans:
            strong_tag = span.find("strong")
            if strong_tag is None:
                continue
            label = strong_tag.get_text(strip=True)

            parent_div = span.parent
            p_tags = parent_div.find_all("p")
            if len(p_tags) == 0:
                continue
            # not active right now
            elif len(p_tags) == 1:
                career_value = p_tags[0].get_text(strip=True)
            # active player
            else:
                career_value = p_tags[1].get_text(strip=True)
                
            if label == "PER":
                result["career_per"] = career_value
            elif label == "WS":
                result["career_win_shares"] = career_value
 
    return result

In [64]:
per_game_df = pd.read_csv("player_per_game_raw.csv")
unique_ids = per_game_df['player_id'].dropna().unique()
print("count of unique players:", len(unique_ids))

count of unique players: 963


In [78]:
#test_ids = unique_ids[:5]

bios = []
for pid in tqdm(unique_ids):
    bio = scrape_player_bio(pid)
    bios.append(bio)
    time.sleep(4)
 
bio_df = pd.DataFrame(bios)

100%|██████████| 963/963 [1:22:09<00:00,  5.12s/it]


In [82]:
bio_df.head()

,player_id,name,position,shoots,height_cm,weight_kg,college,draft_year,nba_debut,birth_date,birth_place,career_per,career_win_shares
0,hardeja01,James Harden,Point Guard and Shooting Guard,Left,196,99,Arizona State,2009.0,"October 28, 2009","August 26,1989",California,23.5,182.4
1,bealbr01,Bradley Beal,Shooting Guard,Right,193,93,Florida,2012.0,"October 30, 2012","June 28,1993",Missouri,18.0,58.4
2,lillada01,Damian Lillard,Point Guard,Right,188,90,Weber State,2012.0,"October 31, 2012","July 15,1990",California,22.2,118.4
3,youngtr01,Trae Young,Point Guard,Right,188,74,Oklahoma,2018.0,"October 17, 2018","September 19,1998",Texas,21.3,44.7
4,antetgi01,Giannis Antetokounmpo,"Power Forward, Small Forward, Point Guard, and...",Right,211,110,NaN,2013.0,"October 30, 2013","December 6,1994",Greece,26.1,125.8


In [84]:
bio_df.tail()

,player_id,name,position,shoots,height_cm,weight_kg,college,draft_year,nba_debut,birth_date,birth_place,career_per,career_win_shares
958,cazalma01,Malcolm Cazalon,Shooting Guard,Left,198,83,NaN,NaN,"November 5, 2023","August 27,2001",France,0.0,0.0
959,crutcja01,Jalen Crutcher,Point Guard,Right,185,79,Dayton,NaN,"February 27, 2024","July 18,1999",Tennessee,-12.6,0.0
960,funkan01,Andrew Funk,Shooting Guard,Right,196,90,NaN,NaN,"March 14, 2024","September 21,1999",Pennsylvania,-5.0,-0.1
961,gateska01,Kaiser Gates,Small Forward,Right,203,101,Xavier,NaN,"October 30, 2023","November 8,1996",Georgia,-19.9,-0.1
962,skapidm01,Dmytro Skapintsev,Center,Right,216,117,Cal State Northridge,NaN,"December 23, 2023","May 12,1998",Ukraine,-19.3,0.0


In [86]:
bio_df.to_csv("player_bios.csv", index=False)